# MMIRAG — controlled experiments (OpenAI only, about US\$7.5)

**What this does.** Runs the controlled experiments on the primary questions and on a **larger, public, pathology-specific** question set, using only low-cost OpenAI models:

* **Data:** MedMCQA questions labelled *Pathology* (public and openly licensed; Indian AIIMS and NEET-PG entrance-exam questions; four options), mapped to the five text domains with inspectable keyword rules and **sampled to 250 questions per domain (1,250 in total)**, plus the primary set (250 unique questions, from `data/FINALE.xlsx`).
* **Reader:** `gpt-4.1-mini` (the reader of the earlier MedMCQA run). A weaker reader (`gpt-4o-mini`, about US\$2 more) can be switched on in Section 2. The frontier models are not re-run; their original primary-set answers are included in the comparisons.
* **Conditions:** No-RAG, single-pass RAG, RAG-SEQ with caps 1/2/3 (cap 1 = the one-round gap-aware control), and a decoupled loop in which a different model (`gpt-4o-mini`) decides when to stop.
* **Output:** one Excel workbook with the questions, every answer, and all statistics (confidence intervals, Holm correction, cost per question with variance, headroom and baseline–gain analyses).

**Cost:** about US\$8 in total (measured pilot: US\$0.005 per question, plus about US\$0.40 for the text-relevance labels; Section 6 prints the projection before you spend more). **Time:** about 2 hours. Progress is saved to Google Drive after every question; after a disconnect, re-run the cells and it continues.

**One prompt change, applied to every condition:** the system prompt asks the model to end with `FINAL ANSWER: <letter>`, because the original `[LETTER] [Justification]` format cannot be read reliably (a sentence starting with "A …" is read as answer A). Every reply is also re-read with the repository's old parser so the two can be compared.

Use a GPU runtime (*Runtime → Change runtime type → T4 GPU*) so the knowledge base embeds quickly.

## 1. Setup

In [1]:
!pip -q install openai sentence-transformers faiss-cpu pymupdf datasets openpyxl

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORKDIR = '/content/drive/MyDrive/MMIRAG_R3'     # results, knowledge base and caches live here
os.makedirs(f'{WORKDIR}/kb/pdfs', exist_ok=True)
REPO = '/content/MMIRAG'
if not os.path.exists(REPO):
    !git clone -q https://github.com/youssefAZAMI360/MMIRAG {REPO}
print(sorted(os.listdir(REPO)))

Mounted at /content/drive
['.git', 'README.md', 'data', 'evaluation', 'prompts', 'results', 'retrieval']


In [3]:
# OpenAI key: Colab "Secrets" panel (key icon, left) -> add OPENAI_API_KEY and allow notebook access.
from google.colab import userdata
from getpass import getpass
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except Exception:
    os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')
print('key set')

OPENAI_API_KEY: ··········
key set


## 2. Settings and prompts (prompts are read from the repository)

In [4]:
# price = (input, output, cached input) in US$ per million tokens, OpenAI list prices
READERS = {
    'gpt-4.1-mini': dict(model='gpt-4.1-mini', price=(0.40, 1.60, 0.10)),   # main reader, as in the paper's replication
    # Optional weaker reader for the reader-strength comparison (adds about US$2; gpt-4.1-nano is retired):
    # 'gpt-4o-mini': dict(model='gpt-4o-mini', price=(0.15, 0.60, 0.075)),
}
MAIN_READER = 'gpt-4.1-mini'       # the only reader that also runs the decoupled-assessor condition
ASSESSOR_MODEL, ASSESSOR_PRICE = 'gpt-4o-mini', (0.15, 0.60, 0.075)   # a different model decides when to stop

N_PER_DOMAIN = 250                 # MedMCQA questions per domain (5 domains -> 1,250); total cost about US$7.5
CAP, TOP_K = 3, 5                  # repository README: max 3 iterations, top-5 per retrieval
EMBED_MODEL = 'sentence-transformers/all-mpnet-base-v2'    # repository README (768-d)
CHUNK_WORDS, CHUNK_OVERLAP = 180, 40                         # repository README
WORKERS = 8                        # lower to 4 if you see many rate-limit retries
MAX_USD = 9.00                     # hard spending cap for this notebook: no API call is made beyond it

In [5]:
import re
def _read(name):
    t = open(f'{REPO}/prompts/{name}', encoding='utf-8').read().replace('\r\n', '\n')
    return '\n'.join(l for l in t.split('\n') if not l.lstrip().startswith('#')).strip() + '\n'

SYSTEM_PROMPT = _read('system_prompt.txt').rstrip() + (
    "\n- After the justification, end your reply with a separate last line: FINAL ANSWER: <letter>"
    "\n- If you reply with GAP_IDENTIFIED instead of answering, do not give an answer or a FINAL ANSWER line.\n")
T_NORAG = _read('no_rag_prompt.txt'); T_SP = _read('rag_single_pass_prompt.txt')
T_IT1 = _read('rag_seq_iteration1.txt'); T_ITN = _read('rag_seq_iterationN.txt')

def _fill_q(t, q):
    letters = [L for L in 'ABCDE' if str(q.get(L, '')).strip()]
    opts = '\n'.join(f'{L}. {q[L]}' for L in letters)
    t = re.sub(r'A\. \{option_a\}.*?E\. \{option_e\}[^\n]*', lambda m: opts, t, flags=re.S)
    hist = str(q.get('history', '') or '').strip()
    t = t.replace('Patient History:\n{patient_history}\n\n', '' if not hist else f'Patient History:\n{hist}\n\n')
    return t.replace('{question_stem}', str(q['question']).strip()), letters

def _chunks(ps):
    return '\n\n'.join(p.strip() for p in ps)

def p_norag(q):
    return _fill_q(T_NORAG, q)

def p_single(q, passages):
    t, L = _fill_q(T_SP, q)
    t = re.sub(r'\{retrieved_chunk_1\}.*?\{retrieved_chunk_3\}', lambda m: _chunks(passages), t, flags=re.S)
    return t, L

def p_iter1(q, passages):
    t, L = p_single(q, passages)
    return t.rstrip() + '\n\n' + T_IT1, L

def p_iterN(q, gap, passages):
    t, L = _fill_q(T_ITN, q)
    t = t.replace('{gap_description_from_previous_iteration}', gap)
    t = re.sub(r'\{new_retrieved_chunk_1\}.*?\{new_retrieved_chunk_2\}', lambda m: _chunks(passages), t, flags=re.S)
    return t, L


ASSESSOR_PROMPT = """You are checking whether retrieved evidence is sufficient to answer a
pathology multiple-choice question. Do NOT answer the question.

--- RETRIEVED CONTEXT ---
{context}
--- END RETRIEVED CONTEXT ---

Question:
{question}

Options:
{options}

If the context contains the information needed to choose an option, reply exactly:
SUFFICIENT
Otherwise reply exactly:
GAP_IDENTIFIED: <what to search for next, in one sentence>"""

FORCE_MSG = ("You have reached the maximum number of retrieval rounds. Using the question, the options "
             "and all the context above, choose the single best option now.\n"
             "Format: [LETTER] [Justification], then a last line FINAL ANSWER: <letter>")

_q = dict(question='Example stem?', A='one', B='two', C='three', D='four', E='')
print(SYSTEM_PROMPT); print('-' * 60); print(p_iter1(_q, ['passage 1', 'passage 2'])[0])

You are an expert medical professional specialising in pathology.

Your task is to answer multiple-choice questions about medical pathology.

You must reason step by step through the clinical and pathological evidence
before selecting your answer.

Rules:
- Read the full question stem and all answer options carefully.
- If a pathological image is provided, integrate its visual findings
  into your reasoning.
- Return ONLY a single capital letter (A, B, C, D, or E) as your answer,
  followed by a concise medical justification of 2-4 sentences.
- Do not add any text before the answer letter.
- Format: [LETTER] [Justification]
- After the justification, end your reply with a separate last line: FINAL ANSWER: <letter>
- If you reply with GAP_IDENTIFIED instead of answering, do not give an answer or a FINAL ANSWER line.

------------------------------------------------------------
Answer the following pathology multiple-choice question.
Use the retrieved context below to support your reason

## 3. Knowledge base
Uses the same corpus as the earlier `gpt-4.1-mini` runs (retrieval ablation and cap sweep): every PDF in the Drive folder **`robbins_pdfs`** (it contains `CAP11.pdf`), chunked exactly as those notebooks did (180 words, 40-word overlap, PyMuPDF text), embedded with `all-mpnet-base-v2`, FAISS `IndexFlatL2`. Those runs produced 1,398 chunks; you should see the same number. The index is saved to `MMIRAG_R3/kb/`, so this runs once.

Do **not** add *Robbins and Cotran Review of Pathology* (a question book) to this folder: retrieving from a book of practice questions could leak answers.

In [6]:
import json, glob, re, threading, numpy as np, pandas as pd, faiss, fitz
from sentence_transformers import SentenceTransformer
# The corpus of the earlier gpt-4.1-mini runs (retrieval ablation, cap sweep):
# their notebooks read every PDF in this Drive folder (it contains CAP11.pdf) and produced 1,398 chunks.
KB_FOLDER = '/content/drive/MyDrive/robbins_pdfs'
INDEX_PATH, CHUNKS_PATH = f'{WORKDIR}/kb/index.faiss', f'{WORKDIR}/kb/chunks.json'
os.makedirs(f'{WORKDIR}/kb', exist_ok=True)
embedder = SentenceTransformer(EMBED_MODEL)

def chunk(t, w=CHUNK_WORDS, o=CHUNK_OVERLAP):      # identical to the chunker of the earlier notebooks
    ws = re.sub(r"\s+", " ", t).strip().split(" ")
    if not ws or ws == [""]: return []
    out, step = [], max(1, w - o)
    for s in range(0, len(ws), step):
        out.append(" ".join(ws[s:s + w]))
        if s + w >= len(ws): break
    return out

if os.path.exists(INDEX_PATH) and os.path.exists(CHUNKS_PATH):
    index = faiss.read_index(INDEX_PATH); CHUNKS = json.load(open(CHUNKS_PATH))
    print('loaded saved index')
else:
    pdfs = sorted(glob.glob(os.path.join(KB_FOLDER, '*.pdf')))
    assert pdfs, f'No PDFs found in {KB_FOLDER}'
    CHUNKS = []
    for f in pdfs:
        doc = fitz.open(f); n0 = len(CHUNKS)
        CHUNKS += chunk("\n".join(doc[pg].get_text() or "" for pg in range(len(doc)))); doc.close()
        print(f'{os.path.basename(f)[:60]:60s} -> {len(CHUNKS) - n0} chunks')
    CHUNKS = [c for c in CHUNKS if c.strip()]
    print(f'{len(CHUNKS)} chunks (the earlier runs had 1,398)')
    vecs = embedder.encode(CHUNKS, batch_size=64, show_progress_bar=True, convert_to_numpy=True).astype('float32')
    index = faiss.IndexFlatL2(vecs.shape[1]); index.add(vecs)
    faiss.write_index(index, INDEX_PATH); json.dump(CHUNKS, open(CHUNKS_PATH, 'w'))
    print('index saved to Drive')
assert index.ntotal == len(CHUNKS) and index.d == embedder.get_sentence_embedding_dimension()
print(f'{len(CHUNKS)} chunks, dimension {index.d}')

_rlock = threading.Lock()
def retrieve(query, k=TOP_K, exclude=()):
    """Top-k chunks by L2 distance, skipping chunks already shown in this question."""
    with _rlock:
        q = embedder.encode([query], convert_to_numpy=True).astype('float32')
        _, idx = index.search(q, k + len(exclude))
    out = [CHUNKS[i] for i in idx[0] if i >= 0 and CHUNKS[i] not in exclude]
    return out[:k]

for i, p in enumerate(retrieve('mechanism of foam cell formation in atherosclerotic plaque'), 1):
    print(i, p[:160].replace('\n', ' '), '...')

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

loaded saved index
1398 chunks, dimension 768


/tmp/ipykernel_2876/3202266907.py:36: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert index.ntotal == len(CHUNKS) and index.d == embedder.get_sentence_embedding_dimension()


1 the thickness and size of the plaque. Figure 1.8 Atherosclerosis, microscopic At high magnification the necrotic center of an aortic atheroma shows foam cells ( ...
2 risk factors through adoption of a healthy lifestyle with increased exercise and reduced caloric intake can halt the progression of atherosclerosis, and atherom ...
3 R 1 Blood Vessels 3 Figure 1.4 Atherosclerosis, gross This is about as normal as an adult aorta can be. The intimal surface is quite smooth, with only occasiona ...
4 the harbinger of worse atherosclerosis to come, if plaques continue to enlarge. Atherosclerosis is initiated with endothelial damage and inflammation with leuko ...
5 are prone to rupture, hemorrhage, and thrombosis. Platelets become activated and adhere to sites of endothelial injury, then release cytokines such as platelet- ...


## 4. Question sets
* **Primary (250):** read from `data/FINALE.xlsx`; the 50 duplicated Atherosclerosis rows are removed.
* **MedMCQA (1,250):** questions MedMCQA labels *Pathology* with a single correct answer, mapped to five domains by keyword rules on the question and MedMCQA's topic name (questions matching no domain or more than one are discarded), then 250 per domain sampled with seed 0. MedMCQA's own question ids are saved so the set can be published with the paper. The MedMCQA topic names are kept for the finer-grained analysis in Section 8.

In [7]:
cols = ['number', 'domain', 'question', 'A', 'B', 'C', 'D', 'E', 'answer'] + [f'x{i}' for i in range(12)]
fin = pd.read_excel(f'{REPO}/data/FINALE.xlsx', sheet_name='No images', header=None, skiprows=2, dtype=str)
fin.columns = cols
fin = fin[fin.domain.str.lower() != 'atherosclerosis'].reset_index(drop=True)
fin['id'] = [f'P{i:03d}_{n}' for i, n in enumerate(fin.number)]
fin['answer'] = fin.answer.str.strip().str.upper()
primary = fin[['id', 'domain', 'question', 'A', 'B', 'C', 'D', 'E', 'answer']].fillna('')
assert len(primary) == 250 and not primary.question.duplicated().any()
primary.to_csv(f'{WORKDIR}/clean250.csv', index=False)
print('primary:', primary.domain.value_counts().to_dict())

primary: {'Neoplasia': 60, 'Cellular Injury': 55, 'Immunopathology': 50, 'Cardiovascular': 45, 'Inflammation': 40}


In [8]:
import re
RULES = {   # matched on lower-cased question + MedMCQA topic name; publish these with the paper
    'Cardiovascular':  r'\b(?:heart|cardiac|cardio\w*|myocard\w*|coronary|endocard\w*|pericard\w*|valv\w*|aort\w*|arter\w*|athero\w*|aneurysm\w*|hypertensi\w*|angina|rheumatic fever|vasculitis)',
    'Cellular Injury': r'\b(?:cell injury|cellular injury|cell death|necros\w*|necrotic|apoptos\w*|apoptotic|hypertroph\w*|atroph\w*|metaplas\w*|hyperplas\w*|calcification|free radical\w*|reperfusion|autophag\w*|fatty change|steatosis|hyaline|lipofuscin|pyknosis|karyorrhexis|karyolysis)',
    'Immunopathology': r'\b(?:immun\w*|hypersensitiv\w*|autoimmun\w*|antibod\w*|antigen\w*|transplant\w*|rejection|graft|amyloid\w*|lupus|sle|hla|complement|t cells?|b cells?|mhc|scleroderma|sjogren\w*|immunodeficien\w*|hiv|aids|anaphyla\w*)',
    'Inflammation':    r'\b(?:inflamm\w*|neutrophil\w*|chemota\w*|granulom\w*|exudat\w*|transudat\w*|abscess\w*|pus|wound\w*|healing|repair|fibrosis|scar|mediators?|prostaglandin\w*|leukotriene\w*|histamine|bradykinin|acute phase|opsoni\w*|phagocyt\w*|margination|diapedesis)',
    'Neoplasia':       r'\b(?:tumou?rs?|neoplas\w*|cancer\w*|carcinom\w*|sarcom\w*|oncogen\w*|metasta\w*|malignan\w*|benign|lymphoma\w*|leuka?emia\w*|carcinogen\w*|p53|staging|grading|paraneoplastic|anaplas\w*|dysplas\w*)',
}
from datasets import load_dataset
mm = load_dataset('openlifescienceai/medmcqa', split='train').to_pandas()
mm = mm[(mm.subject_name == 'Pathology') & (mm.choice_type == 'single')].copy()
mm = mm[mm[['opa', 'opb', 'opc', 'opd']].apply(lambda r: all(str(x).strip() for x in r), axis=1)]   # all four options present
txt = (mm.question.fillna('') + ' ' + mm.topic_name.fillna('')).str.lower()
hits = pd.DataFrame({d: txt.str.contains(r, regex=True) for d, r in RULES.items()})
one = hits.sum(axis=1) == 1
mm = mm[one].copy(); mm['domain'] = hits[one].idxmax(axis=1)
mm = mm.drop_duplicates('question')
print('Pathology questions mapped to exactly one domain:', mm.domain.value_counts().to_dict())
mm = pd.concat([g.sample(n=min(N_PER_DOMAIN, len(g)), random_state=0) for _, g in mm.groupby('domain')])
medmcqa = pd.DataFrame({'id': mm.id, 'domain': mm.domain, 'topic': mm.topic_name.fillna(''), 'question': mm.question,
                        'A': mm.opa, 'B': mm.opb, 'C': mm.opc, 'D': mm.opd, 'E': '',
                        'answer': mm.cop.map(lambda c: 'ABCD'[int(c)])})
medmcqa = medmcqa.sample(frac=1, random_state=0).reset_index(drop=True)
medmcqa.to_csv(f'{WORKDIR}/medmcqa_set.csv', index=False)
print('MedMCQA set:', len(medmcqa), medmcqa.domain.value_counts().to_dict())
print('answer letters:', medmcqa.answer.value_counts().to_dict())
print('topics with >= 25 questions:', int((medmcqa.topic.value_counts() >= 25).sum()))

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 85.9MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  936kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.48MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6150 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4183 [00:00<?, ? examples/s]

Pathology questions mapped to exactly one domain: {'Neoplasia': 1358, 'Immunopathology': 807, 'Inflammation': 606, 'Cardiovascular': 406, 'Cellular Injury': 390}
MedMCQA set: 1250 {'Immunopathology': 250, 'Inflammation': 250, 'Cardiovascular': 250, 'Neoplasia': 250, 'Cellular Injury': 250}
answer letters: {'A': 377, 'C': 336, 'B': 312, 'D': 225}
topics with >= 25 questions: 4


## 5. Core code (no changes needed)

In [9]:
import re
def parse_old(response_text):
    """The repository's evaluation/parse_results.py, verbatim (used only for the comparison)."""
    match = re.search(r"(?:^|\n)\s*([A-E])\b", response_text or '')
    if match:
        return match.group(1)
    match = re.search(r"\b([A-E])\b", response_text or '')
    return match.group(1) if match else "PARSE_ERROR"

def parse_new(text, letters):
    """Strict reader: FINAL ANSWER line > 'answer is X' > a letter opening the reply. Never guesses."""
    if not text:
        return None
    t = text.replace('*', '').replace('`', '').replace('#', '')
    m = list(re.finditer(r'FINAL\s+ANSWER\s*[:\-–]?\s*[\[\(]?\s*([A-E])\b', t, re.I))
    if m:
        L = m[-1].group(1).upper()
        return L if L in letters else None
    m = list(re.finditer(r'\b(?:correct\s+)?answer\s*(?:is|:)\s*(?:option\s*)?[\[\(]?\s*([A-E])\b(?![a-z])', t, re.I))
    if m:
        L = m[-1].group(1).upper()
        return L if L in letters else None
    m = re.match(r'\s*[\[\(]?([A-E])[\]\)]?(?:[\.\):\]]|\s*[-–—:]|\s*\n|\s*$|\s+(?=[A-Z]))', t)
    if m and m.group(1) in letters:
        return m.group(1)
    return None


import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from tqdm.auto import tqdm
_wlock = threading.Lock()
KEEP = ('tin', 'tout', 'reasoning', 'cached', 'cost', 'sec', 'served')

def make_cfg(label, model, price, key='OPENAI_API_KEY', base_url=None, temperature=0, max_tokens=None):
    return dict(label=label, model=model, price=price, temperature=temperature, max_tokens=max_tokens,
                client=OpenAI(api_key=os.environ[key], base_url=base_url, timeout=600))

RCFG = {name: make_cfg(name, r['model'], r['price'], max_tokens=600) for name, r in READERS.items()}
ASSESSOR = make_cfg(ASSESSOR_MODEL, ASSESSOR_MODEL, ASSESSOR_PRICE, max_tokens=120)

# ---------------------------------------------------------------- money safety
# 1) Every API reply is saved to Drive (api_cache.jsonl) the moment it arrives. If the run stops for any
#    reason, re-running never pays again for a call that was already made: it is read from the cache.
# 2) SPENT = everything paid so far through this notebook (rebuilt from the cache on every start).
#    No new call is made once SPENT reaches MAX_USD.
# 3) If OpenAI reports that the credit is exhausted, the whole run stops at once (no retry loop).
import hashlib
CACHE_PATH = f'{WORKDIR}/api_cache.jsonl'
_clock = threading.Lock()
CACHE, SPENT = {}, 0.0
if os.path.exists(CACHE_PATH):
    for l in open(CACHE_PATH):
        try:
            e = json.loads(l); CACHE[e['key']] = e['res']; SPENT += e['res']['cost']
        except Exception:
            pass          # a half-written last line after a crash is skipped
print(f'cache: {len(CACHE)} saved API replies, US${SPENT:.2f} spent so far; budget US${MAX_USD:.2f}')

class StopRun(Exception):
    pass
STOP = threading.Event()

def _save_line(path, obj):
    with open(path, 'a') as f:
        f.write(json.dumps(obj) + '\n'); f.flush(); os.fsync(f.fileno())

def chat(cfg, messages):
    global SPENT
    kw = dict(model=cfg['model'], messages=messages)
    if cfg.get('temperature') is not None: kw['temperature'] = cfg['temperature']
    if cfg.get('max_tokens'): kw['max_tokens'] = cfg['max_tokens']
    key = hashlib.sha256(json.dumps(kw, sort_keys=True).encode()).hexdigest()
    if key in CACHE:
        return dict(CACHE[key])
    for attempt in range(8):
        if STOP.is_set():
            raise StopRun('run stopped')
        with _clock:
            if SPENT >= MAX_USD:
                STOP.set(); raise StopRun(f'budget of US${MAX_USD:.2f} reached')
        try:
            t0 = time.time()
            r = cfg['client'].chat.completions.create(**kw)
            u = r.usage
            pin, pout = (u.prompt_tokens or 0), (u.completion_tokens or 0)
            billed_out = max(pout, (getattr(u, 'total_tokens', 0) or 0) - pin)    # includes hidden reasoning tokens
            det = getattr(u, 'completion_tokens_details', None)
            reas = (getattr(det, 'reasoning_tokens', 0) or 0) if det is not None else 0
            pdet = getattr(u, 'prompt_tokens_details', None)
            cached = (getattr(pdet, 'cached_tokens', 0) or 0) if pdet is not None else 0   # OpenAI caches repeated prefixes automatically
            p_in, p_out, p_cached = cfg['price']
            res = dict(text=r.choices[0].message.content or '', tin=pin, tout=billed_out, reasoning=reas, cached=cached,
                       cost=((pin - cached) * p_in + cached * p_cached + billed_out * p_out) / 1e6,
                       sec=time.time() - t0, served=getattr(r, 'model', '') or '')
            with _clock:
                CACHE[key] = res; SPENT += res['cost']
                _save_line(CACHE_PATH, dict(key=key, model=cfg['model'], res=res))
            return dict(res)
        except StopRun:
            raise
        except Exception as e:
            msg = str(e); status = getattr(e, 'status_code', None)
            if 'insufficient_quota' in msg or 'billing' in msg.lower() or status in (401, 402, 403):
                STOP.set()
                raise StopRun('OpenAI says the credit is exhausted or the key is not allowed: ' + msg[:200])
            if status is not None and status < 500 and status not in (408, 409, 429):
                raise            # bad request or wrong model name: retrying will not help
            wait = min(2 ** attempt, 60)
            print(f'  [{cfg["label"]}] retry in {wait}s: {type(e).__name__}: {msg[:120]}'); time.sleep(wait)
    raise RuntimeError('API call failed 8 times')

def gap_text(text):
    m = re.search(r'GAP_IDENTIFIED\s*:?\s*\[?(.+)', text or '', re.I | re.S)
    return (m.group(1).strip().split('\n')[0].strip(' []') or 'more information needed')[:300] if m else None

def answer_rec(c, letters):
    return dict(ans=parse_new(c['text'], letters), old=parse_old(c['text']), raw=c['text'][:3000], **{k: c[k] for k in KEEP})

def run_question(q, cfg, with_dec=True):
    sysm = {'role': 'system', 'content': SYSTEM_PROMPT}
    rec = dict(id=q['id'], domain=q['domain'], topic=q.get('topic', ''), key=q['answer'], reader=cfg['label'])
    t, letters = p_norag(q)
    rec['norag'] = answer_rec(chat(cfg, [sysm, {'role': 'user', 'content': t}]), letters)
    p1 = retrieve(q['question'])
    t, _ = p_single(q, p1)
    rec['sp'] = answer_rec(chat(cfg, [sysm, {'role': 'user', 'content': t}]), letters)

    # RAG-SEQ as in the repository prompts: iteration 1 = single-pass context + GAP analysis; later
    # iterations add the gap and new context to the same conversation. A forced answer is requested
    # at every round where a gap is flagged, which gives caps 1, 2 and 3 from one trajectory.
    msgs, shown, rounds, gap = [sysm, {'role': 'user', 'content': p_iter1(q, p1)[0]}], list(p1), [], None
    for r in range(1, CAP + 1):
        if r > 1:
            new = retrieve(q['question'] + ' ' + gap, exclude=set(shown)); shown += new
            msgs.append({'role': 'user', 'content': p_iterN(q, gap, new)[0]})
        c = chat(cfg, msgs)
        gap = gap_text(c['text'])
        step = dict(round=r, gap=gap, **answer_rec(c, letters))
        if gap: step['ans'] = None
        if step['ans'] is None:
            fc = chat(cfg, msgs + [{'role': 'assistant', 'content': c['text']}, {'role': 'user', 'content': FORCE_MSG}])
            step['forced'] = answer_rec(fc, letters)
        rounds.append(step)
        if step['ans'] is not None: break
        msgs.append({'role': 'assistant', 'content': c['text']})
        gap = gap or 'more information needed'
    rec['ga'] = rounds

    if with_dec:   # decoupled: the assessor sees all context so far and decides; the reader answers once (SP prompt)
        steps, ctx = [], list(p1)
        for r in range(1, CAP + 1):
            if r > 1:
                ctx += retrieve(q['question'] + ' ' + steps[-1]['gap'], exclude=set(ctx))
            f = dict(context='\n\n'.join(ctx), question=q['question'], options='\n'.join(f'{L}. {q[L]}' for L in letters))
            c = chat(ASSESSOR, [{'role': 'user', 'content': ASSESSOR_PROMPT.format(**f)}])
            g = gap_text(c['text'])
            steps.append(dict(round=r, gap=g, raw=c['text'][:300], **{k: c[k] for k in KEEP}))
            if not g: break
        t, _ = p_single(q, ctx)
        rec['dec'] = dict(steps=steps, stopped_by_cap=bool(steps[-1]['gap']), n_passages=len(ctx),
                          **answer_rec(chat(cfg, [sysm, {'role': 'user', 'content': t}]), letters))
    return rec

def run_sp_only(q, cfg):
    """Single-pass RAG only (used to fill the frontier models' single-pass column on the primary set)."""
    sysm = {'role': 'system', 'content': SYSTEM_PROMPT}
    t, letters = p_single(q, retrieve(q['question']))
    return dict(id=q['id'], domain=q['domain'], key=q['answer'], reader=cfg['label'],
                sp=answer_rec(chat(cfg, [sysm, {'role': 'user', 'content': t}]), letters))

def run_file(df, out_path, fn, workers=None, limit=None):
    """Runs fn on every question not yet in out_path. Each finished question is written to Drive at once.
    Stops cleanly (keeping everything) when the budget is reached or the credit runs out."""
    if STOP.is_set():
        print('Run is stopped (budget or credit). Add credit / raise MAX_USD, then run the "Resume" cell.'); return
    done = set()
    if os.path.exists(out_path):
        for l in open(out_path):
            try: done.add(json.loads(l)['id'])
            except Exception: pass
    todo = [r for r in df.to_dict('records') if r['id'] not in done][: limit or None]
    print(f'{len(done)} already done, {len(todo)} to run -> {os.path.basename(out_path)}')
    failed = []
    ex = ThreadPoolExecutor(workers or WORKERS)
    futs = {ex.submit(fn, q): q['id'] for q in todo}
    try:
        for fu in tqdm(as_completed(futs), total=len(futs)):
            try:
                rec = fu.result()
            except StopRun as e:
                if not STOP.is_set(): STOP.set()
                failed.append(futs[fu]); continue
            except Exception as e:
                failed.append(futs[fu]); print('FAILED', futs[fu], repr(e)[:200]); continue
            with _wlock:
                _save_line(out_path, rec)
            if STOP.is_set():
                break
    finally:
        ex.shutdown(wait=True, cancel_futures=True)
    n_done = sum(1 for _ in open(out_path)) if os.path.exists(out_path) else 0
    print(f'saved: {n_done} questions in {os.path.basename(out_path)} | spent so far US${SPENT:.2f} of US${MAX_USD:.2f}')
    if STOP.is_set():
        print('STOPPED: budget reached or credit exhausted. Everything above is saved on Drive. '
              'Add credit (or raise MAX_USD), then run the "Resume" cell: it continues where it stopped and does not pay twice.')
    elif failed:
        print(len(failed), 'questions failed - re-run the cell to retry them')

def show(r):
    print(f"\n=== {r['id']} ({r['domain']}) key={r['key']} reader={r['reader']}")
    for c in ('norag', 'sp'):
        if c in r: print(f"  {c.upper():5s} new={r[c]['ans']} old={r[c]['old']} served={r[c]['served']} | {r[c]['raw'][:140]!r}")
    for s in r.get('ga', []):
        print(f"  RAG-SEQ round {s['round']}: new={s['ans']} gap={s['gap']!r} forced={s.get('forced', {}).get('ans')} | {s['raw'][:140]!r}")
    for s in r.get('dec', {}).get('steps', []):
        print(f"  ASSESSOR round {s['round']}: gap={s['gap']!r} | {s['raw'][:80]!r}")
    if 'dec' in r: print(f"  DEC new={r['dec']['ans']} ({r['dec']['n_passages']} passages)")

cache: 0 saved API replies, US$0.00 spent so far; budget US$9.00


## 6. Smoke test and cost projection — run this, CHECK it, and only then go on
Runs 20 MedMCQA questions per reader. It prints three examples (raw reply, the letter read by the new parser `new` and by the old parser `old`) and the **projected cost of the full run**. Check that `new` is the model's real answer and that gap-analysis replies are recognised as gaps.

In [12]:
from IPython.display import display
PLAN = [(reader, ds) for reader in READERS for ds in ('primary', 'medmcqa')]
# random run order, so that a run stopped early still covers every domain evenly
SETS = {'primary': primary.sample(frac=1, random_state=0), 'medmcqa': medmcqa}
def job(reader, ds):
    cfg = RCFG[reader]; dec = reader == MAIN_READER
    return lambda q: run_question(q, cfg, with_dec=dec)

proj, total = [], 0
for reader in READERS:
    p = f'{WORKDIR}/TEST_{reader}.jsonl'
    if os.path.exists(p): os.remove(p)
    run_file(medmcqa, p, job(reader, 'medmcqa'), limit=20)
    recs = [json.loads(l) for l in open(p)] if os.path.exists(p) else []
    if not recs:
        print(f'!! {reader}: no successful answers in the pilot - check the error above (model name, credit) before going on'); continue
    for r in recs[:3 if reader == MAIN_READER else 1]: show(r)
    def qcost(r):
        c = r['norag']['cost'] + r['sp']['cost'] + sum(s['cost'] + s.get('forced', {}).get('cost', 0) for s in r['ga'])
        if 'dec' in r: c += r['dec']['cost'] + sum(s['cost'] for s in r['dec']['steps'])
        return c
    per_q = np.mean([qcost(r) for r in recs])
    n_all = len(primary) + len(medmcqa); total += per_q * n_all
    unp = sum(r['norag']['ans'] is None for r in recs) + sum(r['sp']['ans'] is None for r in recs)
    proj.append(dict(reader=reader, usd_per_question=round(per_q, 5), questions=n_all, projected_usd=round(per_q * n_all, 2),
                     unparsed_in_pilot=unp, pilot_acc_norag=round(np.mean([r['norag']['ans'] == r['key'] for r in recs]), 2)))
display(pd.DataFrame(proj))
print(f'Projected total for the full run: US${total + 0.40:.2f} (including about US$0.40 for the text-relevance labels)')

0 already done, 20 to run -> TEST_gpt-4.1-mini.jsonl


  0%|          | 0/20 [00:00<?, ?it/s]

saved: 20 questions in TEST_gpt-4.1-mini.jsonl | spent so far US$0.11 of US$9.00

=== efdc3e56-bd78-4462-9b5f-189c20d03c72 (Neoplasia) key=B reader=gpt-4.1-mini
  NORAG new=B old=B served=gpt-4.1-mini-2025-04-14 | 'B Mixed cell parotid neoplasm, also known as pleomorphic adenoma, is the most common type of salivary gland tumor. It is a benign mixed tumo'
  SP    new=B old=B served=gpt-4.1-mini-2025-04-14 | 'B Pleomorphic adenoma, also known as mixed tumor, is the most common salivary gland tumor, accounting for 65% of all salivary gland tumors. '
  RAG-SEQ round 1: new=B gap=None forced=None | 'B The retrieved context states that pleomorphic adenomas, also known as mixed tumors, are the most common salivary gland tumors, accounting '
  ASSESSOR round 1: gap=None | 'SUFFICIENT'
  DEC new=B (5 passages)

=== e34785ee-5752-4c10-bcbc-8c8f62a28297 (Cardiovascular) key=D reader=gpt-4.1-mini
  NORAG new=D old=D served=gpt-4.1-mini-2025-04-14 | 'D Endothelial injury is the primary initiating e

,reader,usd_per_question,questions,projected_usd,unparsed_in_pilot,pilot_acc_norag
0,gpt-4.1-mini,0.00561,1500,8.41,0,0.8


Projected total for the full run: US$8.81 (including about US$0.40 for the text-relevance labels)


## 7. Full runs
**Nothing is lost if the run stops** (credit exhausted, budget cap reached, Colab disconnects):
* every API reply is written to `MMIRAG_R3/api_cache.jsonl` on Drive the moment it arrives, and every finished question to `r3_<reader>_<set>.jsonl`;
* if OpenAI reports that the credit is exhausted, or `MAX_USD` is reached, the run stops cleanly instead of retrying;
* resuming never pays twice: calls already made are read back from the cache;
* the questions run in random order, so a partial run is still a fair sample of every domain, and Sections 8–11 work on whatever has been saved.

In [13]:
# Text-retrieval relevance on the 250 unique primary questions (an earlier Hit@k analysis used the
# 300-row file and its per-question labels were not saved). Same judge model (gpt-4.1-mini), now shown
# the question, the text of the correct option and one passage. About US$0.40.
EVAL_K = 10
JUDGE = make_cfg('judge', READERS[MAIN_READER]['model'], READERS[MAIN_READER]['price'], max_tokens=5)
JUDGE_SYS = 'You are a medical information-retrieval relevance judge. Reply exactly RELEVANT or NOT.'
def relevance_job(q):
    hits = retrieve(q['question'], k=EVAL_K)
    flags = []
    for p_ in hits:
        c = chat(JUDGE, [{'role': 'system', 'content': JUDGE_SYS},
                         {'role': 'user', 'content': f"QUESTION:\n{q['question']}\n\nCORRECT ANSWER: {q['answer']}. {q[q['answer']]}\n\nPASSAGE:\n{p_[:1200]}"}])
        flags.append(int(c['text'].strip().upper().lstrip('*"\' ').startswith('RELEVANT')))
    return dict(id=q['id'], domain=q['domain'], flags=flags)

def run_all():
    for reader, ds in PLAN:
        print(f'\n######## {reader} / {ds}')
        run_file(SETS[ds], f'{WORKDIR}/r3_{reader}_{ds}.jsonl', job(reader, ds))
    print('\n######## text-retrieval relevance / primary')
    run_file(SETS['primary'], f'{WORKDIR}/r3_relevance_primary.jsonl', relevance_job)
    print('\nSaved on Drive:')
    for reader, ds in PLAN:
        p = f'{WORKDIR}/r3_{reader}_{ds}.jsonl'
        n = sum(1 for _ in open(p)) if os.path.exists(p) else 0
        print(f'  {os.path.basename(p):40s} {n:5d} of {len(SETS[ds])} questions')
    print(f'  api_cache.jsonl: {len(CACHE)} API replies | spent US${SPENT:.2f} of US${MAX_USD:.2f}')
run_all()


######## gpt-4.1-mini / primary
0 already done, 250 to run -> r3_gpt-4.1-mini_primary.jsonl


  0%|          | 0/250 [00:00<?, ?it/s]

saved: 250 questions in r3_gpt-4.1-mini_primary.jsonl | spent so far US$1.59 of US$9.00

######## gpt-4.1-mini / medmcqa
0 already done, 1250 to run -> r3_gpt-4.1-mini_medmcqa.jsonl


  0%|          | 0/1250 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [14]:
STOP.clear()
SETS['medmcqa'] = medmcqa.groupby('domain', sort=False).head(200)   # 1,000 questions; those already answered are kept
SETS['medmcqa'].to_csv(f'{WORKDIR}/medmcqa_set_1000.csv', index=False)
MAX_USD = 8.60
run_all()


######## gpt-4.1-mini / primary
250 already done, 0 to run -> r3_gpt-4.1-mini_primary.jsonl


0it [00:00, ?it/s]

saved: 250 questions in r3_gpt-4.1-mini_primary.jsonl | spent so far US$5.37 of US$8.60

######## gpt-4.1-mini / medmcqa
664 already done, 336 to run -> r3_gpt-4.1-mini_medmcqa.jsonl


  0%|          | 0/336 [00:00<?, ?it/s]

saved: 1000 questions in r3_gpt-4.1-mini_medmcqa.jsonl | spent so far US$7.30 of US$8.60

######## text-retrieval relevance / primary
0 already done, 250 to run -> r3_relevance_primary.jsonl


  0%|          | 0/250 [00:00<?, ?it/s]

saved: 250 questions in r3_relevance_primary.jsonl | spent so far US$7.73 of US$8.60

Saved on Drive:
  r3_gpt-4.1-mini_primary.jsonl              250 of 250 questions
  r3_gpt-4.1-mini_medmcqa.jsonl             1000 of 1000 questions
  api_cache.jsonl: 11889 API replies | spent US$7.73 of US$8.60


### Resume (only if the run stopped)
If it stopped because the credit ran out: add credit on platform.openai.com, then run this cell. If it stopped at the budget cap and you want to continue, raise `MAX_USD` here first. It continues with the questions not yet done and does not pay again for calls already made. After a Colab disconnect, run Sections 1–5 again (they are free: the knowledge base and questions are loaded from Drive) and then this cell.

In [ ]:
MAX_USD = 9.00      # raise only if you added credit and want to spend more
STOP.clear()
run_all()

## 8. Analysis
For every reader and question set: accuracy (Wilson 95% CI), unparsed replies, rounds, cap rate, tokens, cost (bootstrap CI and 5th–95th percentile per question), latency, McNemar contrasts with Holm adjustment (prompt effect, iteration effect, decoupled assessor, RAG vs No-RAG), cost per extra correct answer, per-domain results, and the old-vs-new parser comparison.

In [15]:
from scipy.stats import binomtest, pearsonr
from IPython.display import display

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z*z/n; c = (p + z*z/(2*n)) / d; h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return max(0.0, c - h), min(1.0, c + h)
def mcnemar(x, y):
    b = int(((x == 0) & (y == 1)).sum()); c = int(((x == 1) & (y == 0)).sum())
    return b, c, (binomtest(min(b, c), b + c, 0.5).pvalue if b + c else 1.0)
def holm(ps):
    ps = np.asarray(ps, float); m = len(ps); adj = np.empty(m); run = 0
    for i, j in enumerate(np.argsort(ps)):
        run = max(run, (m - i) * ps[j]); adj[j] = min(1.0, run)
    return adj
def boot_ci(x, B=5000):
    r = np.random.default_rng(0); x = np.asarray(x)
    return np.percentile([r.choice(x, len(x)).mean() for _ in range(B)], [2.5, 97.5])
def cap_view(ga, k):
    cost = tok = sec = 0.0
    for s in ga[:k]:
        cost += s['cost']; tok += s['tin'] + s['tout']; sec += s['sec']
        if s['ans'] is not None:
            return s['ans'], s['old'], s['round'], cost, tok, sec, False
        if s['round'] == k or s is ga[-1]:
            f = s.get('forced') or {}
            return (f.get('ans'), f.get('old'), s['round'], cost + f.get('cost', 0),
                    tok + f.get('tin', 0) + f.get('tout', 0), sec + f.get('sec', 0), True)

NAMES = dict(norag='No-RAG', sp='Single-pass RAG', ga1='RAG-SEQ, 1 round (prompt control)', ga2='RAG-SEQ, cap 2',
             ga3='RAG-SEQ, cap 3', dec='Decoupled assessor, cap 3')
CONTRASTS = [('ga1', 'sp', 'Prompt effect: RAG-SEQ 1 round vs single-pass'), ('ga3', 'ga1', 'Iteration effect: cap 3 vs 1 round'),
             ('ga2', 'ga1', 'cap 2 vs 1 round'), ('dec', 'sp', 'Decoupled loop vs single-pass (same answer prompt)'),
             ('ga3', 'dec', 'Coupled (RAG-SEQ) vs decoupled stopping'), ('ga3', 'norag', 'RAG-SEQ vs No-RAG'),
             ('sp', 'norag', 'Single-pass vs No-RAG'), ('dec', 'norag', 'Decoupled vs No-RAG')]

def load_perquestion(path):
    rows = []
    for l in open(path):
        if not l.strip(): continue
        r = json.loads(l)
        row = dict(id=r['id'], domain=r['domain'], topic=r.get('topic', ''), key=r['key'], reader=r.get('reader', ''))
        for c in ('norag', 'sp', 'dec'):
            if c not in r: continue
            x = r[c]
            row.update({c: int(x['ans'] == r['key']), c+'_old': int(x['old'] == r['key']), c+'_letter': x['ans'], c+'_oldletter': x['old'],
                        c+'_unparsed': x['ans'] is None, c+'_cost': x['cost'], c+'_tok': x['tin'] + x['tout'], c+'_sec': x['sec'],
                        c+'_rounds': 1, c+'_capped': False, c+'_served': x.get('served', '')})
        if 'dec' in r:
            d = r['dec']
            row.update(dec_rounds=len(d['steps']), dec_capped=d['stopped_by_cap'],
                       dec_cost=d['cost'] + sum(s['cost'] for s in d['steps']),
                       dec_tok=d['tin'] + d['tout'] + sum(s['tin'] + s['tout'] for s in d['steps']),
                       dec_sec=d['sec'] + sum(s['sec'] for s in d['steps']))
        if 'ga' in r:
            for k in (1, 2, 3):
                a, o, used, cost, tok, sec, capped = cap_view(r['ga'], k)
                row.update({f'ga{k}': int(a == r['key']), f'ga{k}_old': int(o == r['key']), f'ga{k}_letter': a, f'ga{k}_oldletter': o,
                            f'ga{k}_unparsed': a is None, f'ga{k}_rounds': used, f'ga{k}_capped': capped,
                            f'ga{k}_cost': cost, f'ga{k}_tok': tok, f'ga{k}_sec': sec})
        rows.append(row)
    return pd.DataFrame(rows)

def analyse(path, label=None):
    D = load_perquestion(path); n = len(D)
    conds = [c for c in NAMES if c in D.columns]
    print(f'\n################ {label or os.path.basename(path)}: {n} questions; models that answered: '
          f"{sorted(set(D.get('norag_served', D.get('sp_served', pd.Series(['?']))).dropna()))}")
    S = []
    for c in conds:
        k = int(D[c].sum()); lo, hi = wilson(k, n); cost = D[c+'_cost'] * 1000; ci = boot_ci(cost)
        S.append(dict(reader=label, condition=NAMES[c], n=n, correct=k, accuracy=round(k/n, 3), ci95=f'{lo:.3f}-{hi:.3f}',
                      unparsed=int(D[c+'_unparsed'].sum()), mean_rounds=round(D[c+'_rounds'].mean(), 2),
                      at_cap_pct=round(100*D[c+'_capped'].mean(), 1), tokens=round(D[c+'_tok'].mean()), tokens_sd=round(D[c+'_tok'].std()),
                      usd_per_1000=round(cost.mean(), 3), usd_ci=f'{ci[0]:.3f}-{ci[1]:.3f}',
                      usd_p5_p95=f'{np.percentile(cost, 5):.2f}-{np.percentile(cost, 95):.2f}', sec=round(D[c+'_sec'].mean(), 2)))
    S = pd.DataFrame(S); display(S)
    P = []
    for c in conds:
        wrong_old = D[D[c+'_old'] == 0]
        P.append(dict(condition=NAMES[c], acc_new_parser=round(D[c].mean(), 3), acc_old_parser=round(D[c+'_old'].mean(), 3),
                      letters_disagree=int((D[c+'_letter'] != D[c+'_oldletter']).sum()),
                      old_parser_errors_that_are_A=round((wrong_old[c+'_oldletter'] == 'A').mean(), 2)))
    print('Old vs new parser on the same replies:'); display(pd.DataFrame(P))
    if 'ga3' in D:
        g = D[['ga3_rounds', 'ga3_capped']]
        print('RAG-SEQ cap 3: answered after round 1/2/3 = ' + ' / '.join(
            f"{100*((g.ga3_rounds == r) & ~g.ga3_capped).mean():.1f}%" for r in (1, 2, 3)) +
              f'; still flagging a gap at the cap = {100*g.ga3_capped.mean():.1f}%')
        print(f'Single-pass accuracy, capped vs early-answered: {D.sp[D.ga3_capped].mean():.3f} (n={int(D.ga3_capped.sum())}) vs {D.sp[~D.ga3_capped].mean():.3f}')
        e = n - D.norag.sum()
        print(f'Net share of No-RAG errors corrected by RAG-SEQ (cap 3): {D.ga3.sum() - D.norag.sum()}/{e} = {(D.ga3.sum() - D.norag.sum())/max(e, 1):.1%}')
    if 'dec' in D:
        print(f'Decoupled: assessor stopped after round 1 for {100*(D.dec_rounds == 1).mean():.1f}%, at the cap for {100*D.dec_capped.mean():.1f}%')
    R = []
    for a, b, lab in CONTRASTS:
        if a in D and b in D:
            nb, nc, p = mcnemar(D[b].values, D[a].values)
            R.append(dict(reader=label, contrast=lab, diff_pp=round(100*(D[a].mean() - D[b].mean()), 1), gained=nb, lost=nc, p=p))
    R = pd.DataFrame(R); R['p_holm'] = holm(R.p); display(R)
    for a, b in [('ga3', 'sp'), ('ga3', 'norag'), ('dec', 'sp'), ('ga1', 'sp')]:
        if a in D and b in D:
            dacc = D[a].mean() - D[b].mean(); dcost = D[a+'_cost'].mean() - D[b+'_cost'].mean()
            per = f'US${dcost/dacc:.4f} per extra correct answer' if dacc > 0 else 'no accuracy gain'
            print(f'{NAMES[a]} vs {NAMES[b]}: +US${1000*dcost:.2f} per 1,000 q for {100*dacc:+.1f} pp -> {per}')
    dom = []
    for d_, G in D.groupby('domain'):
        m = len(G); row = dict(reader=label, domain=d_, n=m)
        for c in conds:
            lo, hi = wilson(G[c].sum(), m); row[NAMES[c]] = f'{G[c].mean():.2f} ({lo:.2f}-{hi:.2f})'
        dom.append(row)
    dom = pd.DataFrame(dom); display(dom)
    base = path.rsplit('.', 1)[0]
    S.to_csv(base + '_summary.csv', index=False); R.to_csv(base + '_contrasts.csv', index=False)
    dom.to_csv(base + '_domains.csv', index=False); D.to_csv(base + '_perquestion.csv', index=False)
    return D, S, R, dom

RESULTS = {}
for reader in READERS:
    for ds in ('medmcqa', 'primary'):
        p = f'{WORKDIR}/r3_{reader}_{ds}.jsonl'
        if os.path.exists(p): RESULTS[(reader, ds)] = analyse(p, f'{reader} / {ds}')


################ gpt-4.1-mini / medmcqa: 1000 questions; models that answered: ['gpt-4.1-mini-2025-04-14']


,reader,condition,n,correct,accuracy,ci95,unparsed,mean_rounds,at_cap_pct,tokens,tokens_sd,usd_per_1000,usd_ci,usd_p5_p95,sec
0,gpt-4.1-mini / medmcqa,No-RAG,1000,888,0.888,0.867-0.906,7,1.00,0.0,356,49,0.254,0.252-0.257,0.20-0.33,1.17
1,gpt-4.1-mini / medmcqa,Single-pass RAG,1000,874,0.874,0.852-0.893,7,1.00,0.0,1871,168,0.870,0.866-0.875,0.77-1.01,1.25
2,gpt-4.1-mini / medmcqa,"RAG-SEQ, 1 round (prompt control)",1000,884,0.884,0.863-0.902,1,1.00,62.6,3263,1073,1.280,1.252-1.308,0.47-1.96,1.76
3,gpt-4.1-mini / medmcqa,"RAG-SEQ, cap 2",1000,896,0.896,0.876-0.913,0,1.63,5.3,4474,2168,1.742,1.692-1.793,0.47-2.95,1.87
4,gpt-4.1-mini / medmcqa,"RAG-SEQ, cap 3",1000,898,0.898,0.878-0.915,0,1.68,2.3,4682,2808,1.802,1.741-1.865,0.47-3.36,1.89
5,gpt-4.1-mini / medmcqa,"Decoupled assessor, cap 3",1000,884,0.884,0.863-0.902,5,2.41,63.5,10823,4729,2.527,2.470-2.583,1.03-3.52,3.34


Old vs new parser on the same replies:


,condition,acc_new_parser,acc_old_parser,letters_disagree,old_parser_errors_that_are_A
0,No-RAG,0.888,0.880,16,0.32
1,Single-pass RAG,0.874,0.873,8,0.27
2,"RAG-SEQ, 1 round (prompt control)",0.884,0.882,3,0.30
3,"RAG-SEQ, cap 2",0.896,0.894,6,0.33
4,"RAG-SEQ, cap 3",0.898,0.895,7,0.34
5,"Decoupled assessor, cap 3",0.884,0.883,7,0.25


RAG-SEQ cap 3: answered after round 1/2/3 = 37.4% / 57.3% / 3.0%; still flagging a gap at the cap = 2.3%
Single-pass accuracy, capped vs early-answered: 0.522 (n=23) vs 0.882
Net share of No-RAG errors corrected by RAG-SEQ (cap 3): 10/112 = 8.9%
Decoupled: assessor stopped after round 1 for 25.4%, at the cap for 63.5%


,reader,contrast,diff_pp,gained,lost,p,p_holm
0,gpt-4.1-mini / medmcqa,Prompt effect: RAG-SEQ 1 round vs single-pass,1.0,22,12,0.121449,0.485798
1,gpt-4.1-mini / medmcqa,Iteration effect: cap 3 vs 1 round,1.4,23,9,0.020062,0.160493
2,gpt-4.1-mini / medmcqa,cap 2 vs 1 round,1.2,22,10,0.050102,0.350717
3,gpt-4.1-mini / medmcqa,Decoupled loop vs single-pass (same answer pro...,1.0,24,14,0.143307,0.485798
4,gpt-4.1-mini / medmcqa,Coupled (RAG-SEQ) vs decoupled stopping,1.4,31,17,0.059463,0.356780
5,gpt-4.1-mini / medmcqa,RAG-SEQ vs No-RAG,1.0,34,24,0.237047,0.485798
6,gpt-4.1-mini / medmcqa,Single-pass vs No-RAG,-1.4,21,35,0.081427,0.407134
7,gpt-4.1-mini / medmcqa,Decoupled vs No-RAG,-0.4,30,34,0.707981,0.707981


... (6 outputs left out)

RAG-SEQ cap 3: answered after round 1/2/3 = 24.0% / 75.2% / 0.4%; still flagging a gap at the cap = 0.4%
Single-pass accuracy, capped vs early-answered: 0.000 (n=1) vs 0.912
Net share of No-RAG errors corrected by RAG-SEQ (cap 3): 3/20 = 15.0%
Decoupled: assessor stopped after round 1 for 32.8%, at the cap for 49.2%


,reader,contrast,diff_pp,gained,lost,p,p_holm
0,gpt-4.1-mini / primary,Prompt effect: RAG-SEQ 1 round vs single-pass,0.8,2,0,0.500000,1.0
1,gpt-4.1-mini / primary,Iteration effect: cap 3 vs 1 round,1.6,4,0,0.125000,1.0
2,gpt-4.1-mini / primary,cap 2 vs 1 round,1.6,4,0,0.125000,1.0
3,gpt-4.1-mini / primary,Decoupled loop vs single-pass (same answer pro...,1.2,4,1,0.375000,1.0
4,gpt-4.1-mini / primary,Coupled (RAG-SEQ) vs decoupled stopping,1.2,4,1,0.375000,1.0
5,gpt-4.1-mini / primary,RAG-SEQ vs No-RAG,1.2,6,3,0.507812,1.0
6,gpt-4.1-mini / primary,Single-pass vs No-RAG,-1.2,3,6,0.507812,1.0
7,gpt-4.1-mini / primary,Decoupled vs No-RAG,0.0,6,6,1.000000,1.0


RAG-SEQ, cap 3 vs Single-pass RAG: +US$0.99 per 1,000 q for +2.4 pp -> US$0.0414 per extra correct answer
RAG-SEQ, cap 3 vs No-RAG: +US$1.59 per 1,000 q for +1.2 pp -> US$0.1329 per extra correct answer
Decoupled assessor, cap 3 vs Single-pass RAG: +US$1.49 per 1,000 q for +1.2 pp -> US$0.1243 per extra correct answer
RAG-SEQ, 1 round (prompt control) vs Single-pass RAG: +US$0.51 per 1,000 q for +0.8 pp -> US$0.0638 per extra correct answer


,reader,domain,n,No-RAG,Single-pass RAG,"RAG-SEQ, 1 round (prompt control)","RAG-SEQ, cap 2","RAG-SEQ, cap 3","Decoupled assessor, cap 3"
0,gpt-4.1-mini / primary,Cardiovascular,45,0.84 (0.71-0.92),0.82 (0.69-0.91),0.82 (0.69-0.91),0.87 (0.74-0.94),0.87 (0.74-0.94),0.87 (0.74-0.94)
1,gpt-4.1-mini / primary,Cellular Injury,55,0.93 (0.83-0.97),0.93 (0.83-0.97),0.93 (0.83-0.97),0.93 (0.83-0.97),0.93 (0.83-0.97),0.93 (0.83-0.97)
2,gpt-4.1-mini / primary,Immunopathology,50,0.96 (0.87-0.99),0.96 (0.87-0.99),0.96 (0.87-0.99),0.98 (0.90-1.00),0.98 (0.90-1.00),0.98 (0.90-1.00)
3,gpt-4.1-mini / primary,Inflammation,40,0.97 (0.87-1.00),0.97 (0.87-1.00),0.97 (0.87-1.00),1.00 (0.91-1.00),1.00 (0.91-1.00),0.97 (0.87-1.00)
4,gpt-4.1-mini / primary,Neoplasia,60,0.90 (0.80-0.95),0.87 (0.76-0.93),0.90 (0.80-0.95),0.90 (0.80-0.95),0.90 (0.80-0.95),0.87 (0.76-0.93)


### 8b. Text-retrieval relevance on the clean 250
Hit@1/5/10, MRR and nDCG@10 against the LLM relevance labels, and single-pass RAG accuracy (top-5) with and without a relevant passage in the top 10.

In [16]:
from scipy.stats import fisher_exact
p = f'{WORKDIR}/r3_relevance_primary.jsonl'
if os.path.exists(p):
    Rl = pd.DataFrame([json.loads(l) for l in open(p) if l.strip()])
    F = np.array([f + [0] * (EVAL_K - len(f)) for f in Rl['flags']])
    first = [int(np.argmax(f)) + 1 if f.any() else None for f in F]
    dcg = (F / np.log2(np.arange(2, EVAL_K + 2))).sum(1)
    idcg = np.array([(1 / np.log2(np.arange(2, int(k) + 2))).sum() if k else 1 for k in F.sum(1)])
    rel = dict(n=len(Rl), hit1=F[:, :1].any(1).mean(), hit5=F[:, :5].any(1).mean(), hit10=F.any(1).mean(),
               mrr=np.mean([1 / r if r else 0 for r in first]), ndcg10=np.mean(np.where(F.any(1), dcg / idcg, 0)),
               mean_relevant=F.sum(1).mean(), share_none=1 - F.any(1).mean())
    print({k: round(float(v), 3) for k, v in rel.items()})
    sp_file = f'{WORKDIR}/r3_{MAIN_READER}_primary.jsonl'
    if os.path.exists(sp_file):
        D = load_perquestion(sp_file)[['id', 'sp']].merge(Rl.assign(any_rel=F.any(1).astype(int))[['id', 'any_rel']], on='id')
        a, b = D[D.any_rel == 1].sp, D[D.any_rel == 0].sp
        _, pf = fisher_exact([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]])
        print(f'single-pass accuracy with a relevant passage: {a.mean():.3f} (n={len(a)}); without: {b.mean():.3f} (n={len(b)}); Fisher p = {pf:.3f}')
        rel.update(acc_with_relevant=a.mean(), n_with=len(a), acc_without=b.mean(), n_without=len(b), fisher_p=pf)
    pd.DataFrame([rel]).to_csv(f'{WORKDIR}/r3_relevance_summary.csv', index=False)

{'n': 250.0, 'hit1': 0.532, 'hit5': 0.752, 'hit10': 0.832, 'mrr': 0.629, 'ndcg10': 0.648, 'mean_relevant': 3.432, 'share_none': 0.168}
single-pass accuracy with a relevant passage: 0.952 (n=208); without: 0.690 (n=42); Fisher p = 0.000


## 9. Readers together: headroom, baseline–gain association, and the Excel workbook
* **Readers on identical questions.** For MedMCQA: `gpt-4.1-nano` and `gpt-4.1-mini`. For the primary set, the three frontier models are added from `FINALE.xlsx` (their original No-RAG and RAG-SEQ answers), so five readers of different strength are compared on the same 250 questions.
* **Baseline–gain association** by domain and, for MedMCQA, by topic (≥ 25 questions). **Read the Oldham and split-half columns, not the raw r**: the raw correlation is pushed towards negative values by regression to the mean (in a test of this notebook with *random* answers it came out at about r = −0.7). The split-half estimate takes the baseline from one random half of each group's questions and the gain from the other half, so the two share no noise (range over 200 random splits). The within-reader columns remove differences between readers.
* **Excel workbook** `MMIRAG_R3_results.xlsx` with all of it.

In [17]:
def finale_reader(col_nr, col_seq):
    F = fin[['id', 'domain', 'answer', col_nr, col_seq]].copy()
    return pd.DataFrame({'id': F.id, 'domain': F.domain, 'topic': '',
                         'norag': (F[col_nr].str.strip().str.upper() == F.answer).astype(int),
                         'ga3': (F[col_seq].str.strip().str.upper() == F.answer).astype(int)})

def collect(ds):
    R = {}
    for reader in READERS:
        p = f'{WORKDIR}/r3_{reader}_{ds}.jsonl'
        if os.path.exists(p): R[reader] = load_perquestion(p)
    if ds == 'primary':
        R.update({'GPT-5 (original run)': finale_reader('x0', 'x3'), 'Grok-4 (original run)': finale_reader('x1', 'x4'),
                  'Gemini-2.5 (original run)': finale_reader('x2', 'x5')})
    return R

def within_perm(T, x, y, B=5000):
    T = T.dropna(subset=[x, y]); xc = T[x] - T.groupby('reader')[x].transform('mean'); yc = T[y] - T.groupby('reader')[y].transform('mean')
    obs = np.corrcoef(xc, yc)[0, 1]; rng = np.random.default_rng(0); cnt = 0
    for _ in range(B):
        yp = T.groupby('reader')[y].transform(lambda s: rng.permutation(s.values))
        cnt += abs(np.corrcoef(xc, yp - T.groupby('reader')[y].transform('mean'))[0, 1]) >= abs(obs) - 1e-12
    return obs, (cnt + 1) / (B + 1)

def headroom(R, level, min_n):
    rows = []
    for name, D in R.items():
        for g, G in D.groupby(level):
            if len(G) < min_n or g == '': continue
            a0, a3 = G.norag.mean(), G.ga3.mean()
            rows.append(dict(reader=name, group=g, n=len(G), baseline=a0, ragseq=a3, gain=a3 - a0,
                             norm_gain=(a3 - a0) / (1 - a0) if a0 < 1 else np.nan, mean_acc=(a0 + a3) / 2))
    return pd.DataFrame(rows)

def split_half(R, level, min_n, reps=200):
    rng = np.random.default_rng(0); rs = []
    for _ in range(reps):
        pts = []
        for name, D in R.items():
            for g, G in D.groupby(level):
                if len(G) < min_n or g == '': continue
                idx = rng.permutation(len(G)); h1, h2 = G.iloc[idx[: len(G) // 2]], G.iloc[idx[len(G) // 2:]]
                pts.append((name, h1.norag.mean(), h2.ga3.mean() - h2.norag.mean()))
        P_ = pd.DataFrame(pts, columns=['reader', 'b', 'g'])
        xc = P_.b - P_.groupby('reader').b.transform('mean'); yc = P_.g - P_.groupby('reader').g.transform('mean')
        rs.append(np.corrcoef(xc, yc)[0, 1])
    return np.median(rs), np.percentile(rs, 2.5), np.percentile(rs, 97.5)

ALL_RS, ALL_H = [], []
for ds in ('primary', 'medmcqa'):
    R = collect(ds)
    if not R: continue
    for name, D in R.items():
        n = len(D); e = n - D.norag.sum()
        ALL_RS.append(dict(dataset=ds, reader=name, n=n, no_rag=round(D.norag.mean(), 3),
                           single_pass=round(D.sp.mean(), 3) if 'sp' in D else np.nan, rag_seq=round(D.ga3.mean(), 3),
                           net_share_of_errors_corrected=round((D.ga3.sum() - D.norag.sum()) / max(e, 1), 3),
                           usd_per_1000_ragseq=round(1000 * D.ga3_cost.mean(), 2) if 'ga3_cost' in D else np.nan))
    for level in (('domain', 'topic') if ds == 'medmcqa' else ('domain',)):
        T = headroom(R, level, 25 if level == 'topic' else 1)
        if len(T) < 4 or T.reader.nunique() < 1: continue
        r_raw = pearsonr(T.baseline, T.gain); r_old = pearsonr(T.mean_acc, T.gain)
        Tn = T.dropna(subset=['norm_gain']); r_norm = pearsonr(Tn.baseline, Tn.norm_gain)
        row = dict(dataset=ds, level=level, points=len(T), readers=T.reader.nunique(), n_per_point=f'{T.n.min()}-{T.n.max()}',
                   r_raw=round(r_raw[0], 2), p_raw=r_raw[1], r_oldham=round(r_old[0], 2), p_oldham=r_old[1],
                   r_normalised_gain=round(r_norm[0], 2), p_normalised=r_norm[1])
        if T.reader.nunique() > 1:
            w_raw = within_perm(T, 'baseline', 'gain'); w_old = within_perm(T, 'mean_acc', 'gain')
            row.update(r_within_reader=round(w_raw[0], 2), p_within_perm=w_raw[1], r_within_oldham=round(w_old[0], 2), p_within_oldham=w_old[1])
        sh = split_half(R, level, 25 if level == 'topic' else 1)
        row.update(r_split_half=round(sh[0], 2), split_half_range=f'{sh[1]:.2f} to {sh[2]:.2f}')
        ALL_H.append(row); T.to_csv(f'{WORKDIR}/r3_headroom_{ds}_{level}.csv', index=False)
RS, H = pd.DataFrame(ALL_RS), pd.DataFrame(ALL_H)
print('Readers on identical questions:'); display(RS)
print('\nBaseline-gain association (gain = RAG-SEQ cap 3 minus No-RAG):'); display(H)

xlsx = f'{WORKDIR}/MMIRAG_R3_results.xlsx'
with pd.ExcelWriter(xlsx, engine='openpyxl') as xw:
    medmcqa.to_excel(xw, sheet_name='medmcqa_questions', index=False)
    RS.to_excel(xw, sheet_name='readers_compared', index=False)
    H.to_excel(xw, sheet_name='baseline_gain', index=False)
    pd.concat([v[1] for v in RESULTS.values()]).to_excel(xw, sheet_name='summary', index=False)
    pd.concat([v[2] for v in RESULTS.values()]).to_excel(xw, sheet_name='contrasts', index=False)
    pd.concat([v[3] for v in RESULTS.values()]).to_excel(xw, sheet_name='domains', index=False)
    for (reader, ds), v in RESULTS.items():
        D = v[0]; keep = ['id', 'domain', 'topic', 'key'] + [c + s for c in NAMES if c in D for s in ('_letter', '')]
        D[keep].to_excel(xw, sheet_name=f'{reader}_{ds}'[:31], index=False)
print('saved', xlsx)

Readers on identical questions:


,dataset,reader,n,no_rag,single_pass,rag_seq,net_share_of_errors_corrected,usd_per_1000_ragseq
0,primary,gpt-4.1-mini,250,0.920,0.908,0.932,0.150,1.92
1,primary,GPT-5 (original run),250,0.928,NaN,0.956,0.389,NaN
2,primary,Grok-4 (original run),250,0.884,NaN,0.924,0.345,NaN
3,primary,Gemini-2.5 (original run),250,0.968,NaN,0.972,0.125,NaN
4,medmcqa,gpt-4.1-mini,1000,0.888,0.874,0.898,0.089,1.80



Baseline-gain association (gain = RAG-SEQ cap 3 minus No-RAG):


,dataset,level,points,readers,n_per_point,r_raw,p_raw,r_oldham,p_oldham,r_normalised_gain,p_normalised,r_within_reader,p_within_perm,r_within_oldham,p_within_oldham,r_split_half,split_half_range
0,primary,domain,20,4,40-60,-0.56,0.009906,-0.39,0.085201,0.36,0.157874,-0.44,0.222356,-0.26,0.490502,-0.03,-0.46 to 0.36
1,medmcqa,domain,5,1,200-200,-0.07,0.905994,0.39,0.519655,0.04,0.944587,NaN,NaN,NaN,NaN,0.28,-0.75 to 0.86


saved /content/drive/MyDrive/MMIRAG_R3/MMIRAG_R3_results.xlsx


## 10. The original MedMCQA results: letter check
Re-reads `data/medmcqa_results.csv` from the repository. If most errors are the letter "A" while non-"A" answers are mostly right, the original answers were mis-read.

In [18]:
m = pd.read_csv(f'{REPO}/data/medmcqa_results.csv')
for c in ('norag', 'rag', 'seq'):
    w = m[m['c_' + c] == 0]; nA = m[m[c] != 'A']
    print(f"{c:6s} accuracy {m['c_'+c].mean():.3f} | errors that are 'A': {(w[c] == 'A').mean():.0%} "
          f"| accuracy when the answer read is not 'A': {(nA[c] == nA.correct).mean():.0%}")

norag  accuracy 0.595 | errors that are 'A': 87% | accuracy when the answer read is not 'A': 86%
rag    accuracy 0.437 | errors that are 'A': 91% | accuracy when the answer read is not 'A': 77%
seq    accuracy 0.824 | errors that are 'A': 77% | accuracy when the answer read is not 'A': 93%


## 11. Download everything as one zip (works at any time, also after a partial run)

In [19]:
import zipfile
from google.colab import files
zp = '/content/MMIRAG_R3_results.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f'{WORKDIR}/r3_*') + glob.glob(f'{WORKDIR}/*.xlsx') +
                    [f'{WORKDIR}/medmcqa_set.csv', f'{WORKDIR}/clean250.csv', f'{WORKDIR}/api_cache.jsonl']):
        if os.path.exists(f): z.write(f, os.path.basename(f)); print('added', os.path.basename(f))
files.download(zp)
# The zip is also kept on Drive, so nothing depends on the browser download:
import shutil; shutil.copy(zp, f'{WORKDIR}/MMIRAG_R3_results.zip'); print('copy saved to Drive:', f'{WORKDIR}/MMIRAG_R3_results.zip')

added MMIRAG_R3_results.xlsx
added api_cache.jsonl
added clean250.csv
added medmcqa_set.csv
added r3_gpt-4.1-mini_medmcqa.jsonl
added r3_gpt-4.1-mini_medmcqa_contrasts.csv
added r3_gpt-4.1-mini_medmcqa_domains.csv
added r3_gpt-4.1-mini_medmcqa_perquestion.csv
added r3_gpt-4.1-mini_medmcqa_summary.csv
added r3_gpt-4.1-mini_primary.jsonl
added r3_gpt-4.1-mini_primary_contrasts.csv
added r3_gpt-4.1-mini_primary_domains.csv
added r3_gpt-4.1-mini_primary_perquestion.csv
added r3_gpt-4.1-mini_primary_summary.csv
added r3_headroom_medmcqa_domain.csv
added r3_headroom_primary_domain.csv
added r3_relevance_primary.jsonl
added r3_relevance_summary.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

copy saved to Drive: /content/drive/MyDrive/MMIRAG_R3/MMIRAG_R3_results.zip
